# GPT-2 inference with Triton

I used GPT-2 small (124M parameters) to test how much Triton and CUDA graphs help FP16 inference.

- **Prefill** processes the whole prompt. I tuned the Triton linear kernel and compared PyTorch and Triton with and without CUDA graphs.
- **Decode** processes one new token per sequence. I first added a preallocated KV cache, then used a CUDA graph to replay each step as the cache grows. I compared PyTorch attention with a Triton kernel.

Choose a GPU runtime in Colab and run the cells from top to bottom. The saved results are from an NVIDIA L4. The benchmarks use random weights and synthetic token IDs.


In [ ]:
from pathlib import Path
import subprocess

repo = Path("/content/gpt2-triton-colab")
if not repo.exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/sudhirpol522/GPT2-Triton.git", str(repo)], check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", "04721a523652bbb03abc2ee5f1327de6ecfe3ecb"], cwd=repo, check=True)

%cd /content/gpt2-triton-colab
%pip install -q -r requirements.txt

import torch
import triton

assert torch.cuda.is_available(), "Select a GPU runtime in Colab and run this cell again."
print(torch.cuda.get_device_name())
print("PyTorch", torch.__version__, "| Triton", triton.__version__)


## Prefill

I started with fixed matrix multiplication tiles. Profiling showed that linear layers took most of the GPU kernel time. Tuning the tiles cut batch 1, 1,024-token prefill from 11.525 ms to 7.049 ms in the tuning run. PyTorch took 8.452 ms for that workload.

The cell below runs 12 prompt shapes and checks the FP16 logits. It compares eager execution and CUDA graph replay for both backends. Graph timing includes the input copy and replay. Triton compiles and tunes kernels during warmup.

[Full tuning notes and tables](https://github.com/sudhirpol522/GPT2-Triton#tuning-the-linear-kernel)


In [11]:
%run benchmark.py


GPU: NVIDIA L4
PyTorch: 2.14.1+cu130 Triton: 3.8.0
(1, 1) PyTorch: eager 5.028 ms, graph 1.369 ms, capture+warmup 42.0 ms, checks passed
(1, 1) Triton: eager 5.695 ms, graph 1.374 ms, capture+warmup 25.9 ms, checks passed
(1, 32) PyTorch: eager 5.411 ms, graph 1.445 ms, capture+warmup 23.9 ms, checks passed
(1, 32) Triton: eager 5.732 ms, graph 1.451 ms, capture+warmup 26.2 ms, checks passed
(1, 128) PyTorch: eager 5.473 ms, graph 1.821 ms, capture+warmup 27.1 ms, checks passed
(1, 128) Triton: eager 5.818 ms, graph 1.761 ms, capture+warmup 28.4 ms, checks passed
(1, 512) PyTorch: eager 5.351 ms, graph 4.210 ms, capture+warmup 37.3 ms, checks passed
(1, 512) Triton: eager 5.972 ms, graph 3.889 ms, capture+warmup 37.2 ms, checks passed
(1, 1024) PyTorch: eager 8.135 ms, graph 8.232 ms, capture+warmup 60.2 ms, checks passed
(1, 1024) Triton: eager 7.532 ms, graph 7.290 ms, capture+warmup 47.6 ms, checks passed
(4, 1) PyTorch: eager 5.349 ms, graph 1.349 ms, capture+warmup 38.0 ms, checks

## Decode

The eager reference appends to a preallocated KV cache and reads the filled prefix. The first profile showed that attention took a larger share of GPU work at batch 8 with a 512-token prompt. A fixed-position graph probe showed that reducing launch overhead could also help.

In the growing-cache benchmark, each replay writes to the next cache slot and advances a GPU position tensor. Both graph variants use the same PyTorch model layers and cache writes. PyTorch SDPA masks the unused slots. The Triton kernel reads the filled prefix in tiles.

Each measurement covers 16 successive decode steps after prefill. The benchmark checks all 128 positions across eight workloads against eager PyTorch. The saved output below is a repeat of the run in the README.

[Decode profiles and results](https://github.com/sudhirpol522/GPT2-Triton#profiling-decode-on-the-l4)


In [23]:
%run benchmark_decode.py


NVIDIA L4 | GPT-2 124M FP16
PyTorch 2.14.1+cu130 | Triton 3.8.0
Batch  Prompt   Eager ms   PT graph   Triton graph   PT/Triton   Max error
    1      32      4.558      1.438          1.332       1.08x    0.003906
    1     128      4.609      1.536          1.395       1.10x    0.003906
    1     512      4.787      1.790          1.548       1.16x    0.003906
    1    1008      4.739      2.091          1.743       1.20x    0.004211
    4     128      4.772      1.561          1.425       1.10x    0.004883
    4     512      4.847      1.910          1.697       1.13x    0.004639
    8     128      4.740      1.635          1.511       1.08x    0.004395
    8     512      4.789      2.240          2.065       1.08x    0.004883
Both graph paths matched eager at all 128 growing-cache positions.


## Tests

These checks cover the KV cache, greedy decoding, and graph replay across an attention tile boundary.


In [24]:
subprocess.run(["python", "-m", "unittest", "-v", "test_decode.py"], check=True)


test_cache_requires_inference (test_decode.DecodeTests.test_cache_requires_inference) ... ok
test_cache_validation (test_decode.DecodeTests.test_cache_validation) ... ok
test_cached_logits_match_full_forward (test_decode.DecodeTests.test_cached_logits_match_full_forward) ... ok
test_decode_graph_advances_past_attention_tile_boundary (test_decode.DecodeTests.test_decode_graph_advances_past_attention_tile_boundary) ... ok
test_greedy_generation_matches_full_forward (test_decode.DecodeTests.test_greedy_generation_matches_full_forward) ... ok

----------------------------------------------------------------------
Ran 5 tests in 2.697s

OK
